# Define functions for Data Quality Checks

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Function to check null columns

In [0]:
def check_not_null (df,columns):
    ''' 
    Checks if the columns in the dataframe are empty or not.
    If it is null or empty then raise ValueError else return None.
    '''
    results ={}
    for column_name in columns:
        null_count = df.filter((col(column_name).isNull()) | (trim(col(column_name)) == '')).count() # count null values for column_name
        if null_count > 0:
            results[column_name] = null_count

    if len(results) == 0:
        print('check_not_null: PASSED')
        return None
    else:
        raise ValueError(f'check_not_null FAILED: Nulls found. The following are the pairs of null value columns and its null counts - {results}')

### Function to check uniqueness of columns

In [0]:
def check_unique (df,columns):
    '''
    Checks if the columns in the dataframe are unique or not. It based on one key or composite keys.
    If it is not unique then raise ValueError else return None.
    '''
    duplicate_df = df.groupBy(*columns).count().filter(col('count') > 1)
    duplicate_count =duplicate_df.count()
    duplicate_keys = duplicate_df.drop('count').take(5)

    if duplicate_count == 0:
        print('check_unique: PASSED')
        return None
    else:
        raise ValueError(f'check_unique FAILED: Duplicates found.The duplicates found on {columns}, total keys duplicated are {duplicate_count} and the some of them are listed here: {duplicate_keys}')

### Verify the row count


In [0]:
def check_row_count(df,bronze_table,min_pct,max_pct):
    '''
    Checks if the silver table has the numbers of rows between our expected range.
    '''
    bronze_count = spark.read.table(bronze_table).count()

    if bronze_count == 0:
        raise ValueError(f"check_row_count FAILED: Bronze table {bronze_table} has 0 rows — cannot compute retention.")

    else:
        silver_count = df.count()
        if (silver_count >= (bronze_count * (min_pct/100))) and (silver_count <= (bronze_count * (max_pct/100))):
            print('check_row_count: PASSED')
            return None
        else:
            actual_pct = (silver_count/bronze_count)*100
            raise ValueError(f'check_row_count FAILED. Bronze count: {bronze_count}, Silver count: {silver_count}, Actual%: {actual_pct:.2f} % and expected range between:  {min_pct} and {max_pct}')
